# 임베딩 실습 2일차: 수산·해양 논문 토픽 모델링과 검증

> 논문 29,417편을 임베딩으로 묶어 **주제(토픽)**를 찾고, 원문·이름·안정성으로 **검증**한 기록입니다.

## 전체 흐름

```
논문 제목 + 초록
   ↓ ① 임베딩 (bge-m3, 1,024차원)           [1일차에 완료]
   ↓ ② UMAP (5차원으로 축소)
   ↓ ③ HDBSCAN (군집 → 토픽, 나머지는 이상치 −1)
   ↓ ④ c-TF-IDF (토픽마다 상위 단어)
   ↓ ⑤ 토픽 표 (건수 · 비율 · 상위 단어 · 대표 문서)
   ↓ ⑥ 검증: 원문 읽기 → 이름 확인 → 시드 안정성
   ↓ ⑦ 코드북 (테마 정의 · 포함 · 제외)
```

## 오늘 배운 핵심 3가지
1. **비율의 분모는 이상치를 포함한 전체 논문 수**입니다.
2. **최소 군집 크기**가 토픽 수와 이상치 비율을 바꿉니다.
3. 토픽은 **원문 · 이름 · 시드 안정성**을 모두 통과해야 테마로 씁니다.

## 1. 준비

- `works.csv`(논문 제목·초록·토픽 라벨)와 `emb_bge_m3.npy`(임베딩)가 필요합니다.
- 런타임이 바뀌었다면 두 파일을 다시 올린 뒤 아래 셀을 실행하세요.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('works.csv').dropna(subset=['title', 'abstract']).reset_index(drop=True)
emb = np.load('emb_bge_m3.npy')
assert len(df) == emb.shape[0]
N = len(df)
print('논문 수:', N, '| 벡터 차원:', emb.shape[1])

## 2. 차원 줄이기 (UMAP)

**왜 하나요?** 1,024차원에서는 가까운 논문과 먼 논문의 거리 차이가 작아 군집이 잘 안 보여요. 그래서 **이웃 관계를 살리면서 5차원**으로 줄인 뒤 묶어요.

- `n_neighbors=15`: 이웃을 몇 개까지 볼지
- `n_components=5`: 줄일 차원 수
- `random_state=0`: 같은 결과가 나오도록 시드 고정

In [ ]:
import umap

red = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=0).fit_transform(emb)
print('UMAP 결과 모양:', red.shape)

## 3. 토픽 만들기 (HDBSCAN)

**왜 하나요?** 군집 수를 미리 정하지 않고, **밀도가 높은 곳**을 토픽으로 골라요. 최소 군집 크기보다 작은 묶음은 이상치(−1)가 돼요.

### 결과 (최소 군집 크기 50)
- 토픽 **40개**, 이상치 **9,239편 (31.41%)**
- 토픽 비율 합 68.59% + 이상치 31.41% = **100%** → 분모가 전체 29,417편으로 맞아요.
- 가장 큰 토픽은 상어·가오리(토픽 3, 16.75%)예요.

In [ ]:
from sklearn.cluster import HDBSCAN

lab = HDBSCAN(min_cluster_size=50).fit_predict(red)

n_out = (lab == -1).sum()
sizes = pd.Series(lab[lab != -1]).value_counts().sort_values(ascending=False)
print('토픽 수:', len(sizes), '| 이상치:', n_out, f'({n_out / N:.2%})')
print('토픽 비율 합 (이상치 제외):', f'{sizes.sum() / N:.2%}')

## 4. 최소 군집 크기 비교

같은 UMAP 결과에서 최소 군집 크기만 바꿔 봤어요.

| 최소 군집 크기 | 토픽 수 | 이상치 비율 | 가장 큰 토픽 비율 | 해석 |
|---:|---:|---:|---:|---|
| 30 | 90 | 43.74% | 11.70% | 너무 잘게 나뉨 |
| **50** | **40** | **31.41%** | **16.75%** | **균형이 좋음 (선택)** |
| 100 | 5 | 0.54% | 68.03% | 한 덩어리로 뭉침 |
| 150 | 4 | 1.36% | 68.03% | 한 덩어리로 뭉침 |

- 100 이상에서는 가장 큰 토픽이 68%예요. 주제가 나뉘지 않은 거예요.
- 그래서 **50**을 골랐어요.

In [ ]:
rows = []
for mcs in [30, 50, 100, 150]:
    l = HDBSCAN(min_cluster_size=mcs).fit_predict(red)
    n_o = (l == -1).sum()
    sz = pd.Series(l[l != -1]).value_counts()
    rows.append([mcs, len(sz), f'{n_o / N:.2%}', f'{sz.max() / N:.2%}' if len(sz) else '-'])

pd.DataFrame(rows, columns=['최소 군집 크기', '토픽 수', '이상치 비율', '가장 큰 토픽 비율'])

## 5. 상위 단어 (c-TF-IDF)

**왜 하나요?** 토픽 번호만으로는 주제를 알 수 없어요. 토픽에 **자주 나오고 다른 토픽에는 드문 단어**를 뽑아 이름 후보로 써요.

```
W = tf × log(1 + A / f)
tf: 토픽 안에서 단어 비율
f : 전체 토픽에서 단어 횟수
A : 토픽 문서 평균 단어 수
```

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = (df['title'] + ' ' + df['abstract']).values
topics = [t for t in sorted(set(lab)) if t != -1]
topic_docs = [' '.join(docs[lab == t]) for t in topics]

cv = CountVectorizer(stop_words='english', token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]{2,}\b')
X = cv.fit_transform(topic_docs).toarray()
words = np.array(cv.get_feature_names_out())
tf = X / X.sum(axis=1, keepdims=True)
f = X.sum(axis=0)
A = X.sum(axis=1).mean()
W = tf * np.log(1 + A / f)

for k, t in enumerate(topics[:10]):
    print(f'토픽 {t} ({(lab == t).sum()}편): ' + ', '.join(words[np.argsort(-W[k])[:5]]))

### 결과 해석 (상위 10개 토픽)
- **영어 주제 토픽**: 3(상어·가오리), 4(연체류), 21(보호구역), 22(공간계획), 24(연안·해양 관리), 25(블루 이코노미), 8(해양 경제) 등
- **언어로 묶인 토픽**: 13(인도네시아어 `ikan, dan, yang`), 36(포르투갈·스페인어 `pesca, que, dos`) 등 → 주제가 아니라 **언어 묶음**이라 테마에서 제외
- **데이터 노이즈**: 1번(`download, gbif, dataset`)은 논문 주제가 아니라 **데이터 출처 문구**라 제외

## 6. 토픽 표 완성

교안의 마지막 결과물인 **토픽 표**예요. 토픽마다 건수, 비율(분모 = 전체 N), 상위 단어, 대표 문서를 넣고, 이상치 행을 맨 아래에 붙여요.

- **대표 문서**: 토픽 중심에 가장 가까운 논문

> 확인할 점: 비율 열의 합 + 이상치 비율 = 100%

In [ ]:
rows = []
for k, t in enumerate(topics):
    idx_t = np.where(lab == t)[0]
    cen = red[idx_t].mean(axis=0)
    rep = idx_t[np.argmin(np.linalg.norm(red[idx_t] - cen, axis=1))]
    rows.append([t, len(idx_t), round(len(idx_t) / N * 100, 2),
                 ', '.join(words[np.argsort(-W[k])[:5]]), df['title'].iloc[rep][:60]])

table = pd.DataFrame(rows, columns=['토픽', '건수', '비율(%)', '상위 단어 5개', '대표 문서'])
table.loc[len(table)] = ['이상치(-1)', n_out, round(n_out / N * 100, 2), '', '']
table = table.sort_values('건수', ascending=False).reset_index(drop=True)
table.head(15)

## 7. 검증 1: 원문 읽기

**규칙**: 대표 1건 + 무작위 3건 + 경계 1건(중심에서 가장 먼 논문)을 읽어요. **다른 사건이 2건 이상이면 "사건이 여럿"**으로 봐요.

### 결과 요약
| 토픽 | 판정 | 메모 |
|---|---|---|
| 3 상어·가오리 | 사건 하나 (4/5) | 경계 1건은 상어 보호 정책 (포함) |
| 4 연체류 | 사건 하나 (4/5) | 경계 1건은 부수 어획물 경제 (제외) |
| 21 보호구역 | 사건 하나 (5/5) | |
| 25 블루 이코노미 | 사건 하나 (5/5) | |
| 8 중국 중심 해양 경제 | 사건 하나 (5/5) | |
| 22 해양공간계획 | 사건 하나 (5/5) | |
| 24 연안·해양 관리 | 사건 하나 (5/5) | 아래 하위 토픽으로 나눠 봄 |
| 38 양식·식량 안보 | 사건 하나 (4/5) | 시장 분석 1건 제외 |

In [ ]:
def read5(t, seed=0):
    idx_t = np.where(lab == t)[0]
    cen = red[idx_t].mean(axis=0)
    d = np.linalg.norm(red[idx_t] - cen, axis=1)
    rep = idx_t[np.argmin(d)]
    rng = np.random.default_rng(seed)
    rand3 = rng.choice(idx_t, 3, replace=False)
    edge = idx_t[np.argmax(d)]
    print(f'===== 토픽 {t} ({len(idx_t)}편) 원문 5건 =====')
    for tag, i in [('대표', rep)] + [('무작위', i) for i in rand3] + [('경계', edge)]:
        print(f"  [{tag}] {df['title'].iloc[i][:90]}")

for t in [3, 4, 21, 25, 8, 22, 38]:
    read5(t)

## 8. 검증 2: 이름 확인 (무작위 10건)

**규칙**: 무작위 10건 중 **8건 이상**이 이름에 맞아야 통과해요. 8건이 기준인 이유는 맞은 비율의 95% 구간 하한이 0.5를 넘는 최소값이기 때문이에요.

### 결과 요약
| 토픽 | 이름 | 맞은 건수 | 판정 |
|---|---|:---:|---|
| 3 | 상어·가오리 연구 | 9/10 | 통과 |
| 4 | 연체류 연구 | 8/10 | 통과 (경계) |
| 21 | 해양보호구역과 보전 | 9/10 | 통과 |
| 25 | 블루 이코노미 | 10/10 | 통과 |
| 8 | 중국 중심 해양 경제와 환경 | 9/10 | 통과 |
| 22 | 해양공간계획 | 10/10 | 통과 |
| 38 | 양식·식량 안보 | 9/10 | 통과 (아래 안정성에서 제외) |

In [ ]:
def pick10_show(t, seed=0):
    idx_t = np.where(lab == t)[0]
    rng = np.random.default_rng(seed)
    pick = rng.choice(idx_t, 10, replace=False)
    print(f'===== 토픽 {t} 이름 확인 10건 =====')
    for k, i in enumerate(pick, 1):
        print(f"  {k:>2}. {df['title'].iloc[i][:100]}")

for t in [3, 4, 21, 25, 8, 22, 38]:
    pick10_show(t)

## 9. 검증 3: 시드 안정성

**규칙**: 난수 시드(1, 2)를 바꿔 다시 묶었을 때, 같은 논문들이 **자카드 계수 0.5 이상**으로 다시 나와야 해요.

- 자카드 계수 = (두 토픽이 함께 가진 논문 수) ÷ (둘 중 하나라도 가진 논문 수)
- 토픽 번호는 시드마다 바뀌므로, 번호가 아니라 **함께 있는 논문**으로 맞춰요.

### 결과
| 토픽 | 시드 1 | 시드 2 | 판정 |
|---|---:|---:|---|
| 3 상어·가오리 | 0.99 | 0.98 | 안정 |
| 4 연체류 | 0.99 | 0.98 | 안정 |
| 21 보호구역 | 0.93 | 0.91 | 안정 |
| 25 블루 이코노미 | 0.91 | 0.89 | 안정 |
| 8 중국 중심 해양 경제 | 0.98 | 0.97 | 안정 |
| 22 해양공간계획 | 0.91 | 0.91 | 안정 |
| 38 양식·식량 안보 | 0.62 | **0.37** | **불안정 → 제외** |
| 24 연안·해양 관리 | 0.11 | 0.11 | **불안정 → 나눠서 봄** |

In [ ]:
import umap
from sklearn.cluster import HDBSCAN

def make_topics(seed):
    r = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=seed).fit_transform(emb)
    return HDBSCAN(min_cluster_size=50).fit_predict(r)

check = [3, 4, 21, 25, 8, 22, 38, 24]
for seed in [1, 2]:
    lab2 = make_topics(seed)
    for t in check:
        A_set = set(np.where(lab == t)[0])
        best, best_j = None, 0
        for u in set(lab2) - {-1}:
            B = set(np.where(lab2 == u)[0])
            j = len(A_set & B) / len(A_set | B)
            if j > best_j:
                best, best_j = u, j
        print(f'시드 {seed} | 토픽 {t} → 대응 {best}, 자카드 {best_j:.2f}')

## 10. 토픽 24 나눠 보기

토픽 24는 시드를 바꾸면 흔들렸어요(자카드 0.11). 그래서 **토픽 24 안만 따로 다시 묶어** 하위 토픽을 봤어요.

### 하위 토픽 결과 (6개, 이상치 612편 = 29.5%)
| 하위 | 편수 | 상위 단어 | 이름 (확인 결과) |
|---|---:|---|---|
| 0 | 480 | marine, pollution, human | 해양 환경 연구와 영향 (넓힌 이름, 8/10 통과) |
| 1 | 110 | water, sea, bay, black, estuary | 내해·하구 수질 연구 (넓힌 이름, 9/10 통과) |
| 2 | 61 | baltic, ecosystem, governance | 발트해 생태와 관리 (10/10 통과) |
| 3 | 237 | ecosystem, EBM, risk, assessment | 생태계 기반 관리와 위험 평가 (10/10 통과) |
| 4 | 318 | ocean, governance, science, policy | 해양 과학과 정책 (10/10 통과) |
| 5 | 254 | coastal, adaptation, climate, planning | 연안 기후 적응과 계획 (8/10 통과) |

> 이름을 처음에 너무 좁게 붙였던 하위 0(해양 오염 6/10), 하위 1(흑해·하구 수질 7/10)은 이름을 넓혀서 다시 확인했어요.

In [ ]:
idx24 = np.where(lab == 24)[0]
e24 = emb[idx24]
r24 = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=0).fit_transform(e24)
sub = HDBSCAN(min_cluster_size=50).fit_predict(r24)

print('하위 토픽 수:', len(set(sub) - {-1}), '| 이상치:', (sub == -1).sum(), f'({(sub == -1).mean():.1%})')
for s in sorted(set(sub) - {-1}):
    print(f'하위 {s}: {(sub == s).sum()}편')

## 11. 최종 테마와 코드북 (v1)

확정된 테마는 **12개**예요. 제외된 것은 아래 "한계" 칸에 정리했어요.

| 테마 | 정의 | 포함 기준 | 제외 기준 |
|---|---|---|---|
| 상어·가오리 연구 | 상어와 가오리류의 분류, 생물학, 개체군, 어획, 보전 | 상어·가오리가 주제. 나이·성장, 어획, 생리·이동, **보호 정책·법** 포함 | 상어가 아닌 어종이 주제(예: tarpon) |
| 연체류 연구 | 문어·오징어·갑오징어의 생물학, 분류, 산업, 어획 | 두족류가 주제. 언어와 무관하게 주제로 판단 | 두족류가 주제가 아닌 논문. 부수 어획물로만 다룬 경제 연구 |
| 해양보호구역과 보전 | 보호구역의 지정, 계획, 운영, 효과와 사회적 영향 | 보호구역 또는 보전 구역이 주제 | 보호구역과 무관한 해양 생물 역사·분류 |
| 블루 이코노미 | 해양 경제 성장의 정책, 재정, 지속가능성, 사회 영향 | 블루 이코노미·블루 성장이 주제 | 블루 이코노미와 무관한 일반 해양 연구 |
| 중국 중심 해양 경제와 환경 | 중국 중심 해양 경제, 해양 환경 관리, 양식 생태 | 해양 경제·환경·양식 생태가 주제 | 해양 경제와 무관한 생물 연구 |
| 해양공간계획 | 해양공간계획(MSP)과 생태계 기반 해양 관리 | 해양공간계획·관리 계획·평가가 주제 | 육상 계획, 해양과 무관한 공간 계획 |
| 발트해 생태와 관리 (24-2) | 발트해의 생태 상태, 생물다양성, 관리·거버넌스 | 발트해가 주제 | 다른 해역이 주제인 연구 |
| 생태계 기반 관리와 위험 평가 (24-3) | 누적 영향, 위험 평가, 생태계 기반 관리 | 생태계 위험·누적 영향·관리 방법이 주제 | 단일 종의 생리·생태만 다룬 연구 |
| 해양 과학과 정책 (24-4) | 해양 거버넌스, 과학 정책, 지식 생산과 참여 | 해양 정책·거버넌스·과학의 역할이 주제 | 해양 환경 자체가 주제인 생태 연구 |
| 연안 기후 적응과 계획 (24-5) | 연안 기후 적응, 해수면 상승 대응, 연안 계획 | 연안 기후 적응·계획·관리가 주제 | 연안과 무관한 관광·경제 연구 |
| 해양 환경 연구와 영향 (24-0) | 해양 오염과 인간 압력이 해양 환경에 주는 영향 | 해양 오염, 연안 환경 압력, 기후의 해양 영향이 주제 | 생물 분류나 교육 자료처럼 환경 영향이 주제가 아닌 논문 |
| 내해·하구 수질 연구 (24-1) | 흑해·카스피해·하구 등 내해와 하구의 수질, 생태 상태 | 내해·하구의 수질, 수생 생태, 모니터링이 주제 | 외해가 주제인 연구, 생태 재난 자체가 주제인 연구 |

## 12. 한계와 해석 주의

**1. 표본 구성**
- 토픽별로 **최대 5,000편**만 받았어요. 그래서 토픽 크기는 연구량이 아니라 **수집 방식**의 영향을 받아요.
- 상어·가오리 묶음(4,928편)의 98%(4,849편)가 "어류·해양생물" 토픽에서 왔어요. 이 표본에서 가장 컸다는 뜻이지, 전 세계 연구에서 가장 많다는 뜻은 아니에요.

**2. 시간 추세**
- 발행 연도 정보를 저장하지 않았어요. 그래서 **연도별 트렌드는 말할 수 없어요.** 트렌드를 보려면 연도별 편수와 전체 대비 비율이 필요해요.

**3. 언어 효과**
- 인도네시아어, 포르투갈어, 스페인어 논문이 **주제가 아니라 언어**로 묶였어요. 이 토픽들은 테마에서 제외했어요.

**4. 이상치**
- 최소 군집 크기 50에서 이상치가 31.41%예요. 비율을 말할 때는 **분모가 전체 29,417편**이라는 걸 함께 밝혀야 해요.

**5. 제외된 토픽**
- **38(양식·식량 안보)**: 시드를 바꾸면 0.37로 흔들려서 제외했어요.
- **24 전체**: 시드를 바꾸면 0.11로 흔들려서, 하위 토픽으로 나눠서만 써요.
- **1(GBIF 다운로드)**: 데이터 출처 문구라서 제외했어요.
- **13, 36 등 언어 토픽**: 주제가 아니라서 제외했어요.

## 13. 팀 라운드용 요약 (쉬운 말)

> 수산·해양 분야 논문 29,417편(OpenAlex, 수산·해양 토픽 6개에서 토픽당 최대 5,000편)을 뜻이 비슷한 것끼리 묶어 주제를 찾았습니다.
>
> - 주제는 40개가 나왔고, 원문 · 이름 · 시드 안정성을 모두 통과한 **12개 테마**를 확정했습니다.
> - 전체의 **31.41%**는 어느 주제에도 속하지 않아 이상치로 두었습니다. 비율의 분모는 전체 논문 수입니다.
> - 가장 큰 묶음은 **상어·가오리 연구**였습니다. 이는 우리 표본(어류·해양생물 토픽 중심)에서의 결과이며, 연구 동향이나 트렌드로 일반화하지 않습니다.
> - 연도별 추이, 전 세계 연구량 순위는 이 자료로 말할 수 없습니다.

### 보내기 전 체크
- [ ] 비율 분모가 전체 29,417편인지
- [ ] "가장 많다"를 표본 결과로 한정했는지
- [ ] 트렌드·순위로 일반화하지 않았는지
- [ ] 재현 설정(임베딩, UMAP, HDBSCAN, 시드)을 분석팀 기록에 남겼는지